In [0]:

# Databricks notebook source
# %md
## Shipment Tracking — medallion pipeline with quality gates
# **Bronze → [GATE] → Silver → [GATE] → Gold → [GATE]**

# COMMAND ----------
# %md ## 0
SCHEMA = "yusen_catalog.default"                               # where tables are written
BASE   = "/Volumes/yusen_catalog/default/datasource"           # this is where the files are written in my DataBricks so please change accoringly
spark.conf.set("spark.sql.session.timeZone", "UTC")            # all times compared in UTC
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

# COMMAND ----------
def verify_data_quality(pipeline_stage, quality_rules):
    """quality_rules: list of (description, validation_query) -> HARD (stops pipeline), or (description, validation_query, 'WARN') -> tracked only.
    HARD = integrity that must never be wrong (grain, FK, uniqueness).
    WARN = quality signals we quarantine instead of halting (e.g. rejected-row counts)."""
    # Keep track of any mandatory (critical) rules that fail
    failed_critical_rules = []
    
    for rule in quality_rules:
        # Extract rule details
        rule_description = rule[0]
        validation_query = rule[1]
        
        # Default to "HARD" severity if not specified
        severity = rule[2] if len(rule) > 2 else "HARD"
        
        # Run the query to count records violating this rule
        invalid_records_count = spark.sql(validation_query).first()[0]
        is_rule_satisfied = (invalid_records_count == 0)
        
        # Determine the status label
        if is_rule_satisfied:
            status_label = "PASS"
        elif severity == "WARN":
            status_label = "WARN"
        else:
            status_label = "FAIL"
            
        # Print a clear diagnostic message to the console
        print(f"[{status_label}] {severity} Stage: {pipeline_stage} · Rule: '{rule_description}' · Found: {invalid_records_count} bad records")
        
        # If the check failed and it's a hard/critical rule, log it as a failure
        if not is_rule_satisfied and severity == "HARD":
            failed_critical_rules.append(rule_description)
            
    # Stop the entire pipeline if any mandatory checks fail
    if failed_critical_rules:
        raise Exception(
            f"DATA QUALITY GATE FAILED for stage '{pipeline_stage}' due to critical issues: {failed_critical_rules}. "
            f"Pipeline STOPPED to prevent corrupt data from moving forward."
        )
        
    print(f"==> DATA QUALITY GATE PASSED for stage '{pipeline_stage}' (all critical checks clean). Safe to continue.")


In [0]:


# COMMAND ----------
# %md 
# ## 2 · BRONZE — warehouse (BATCH, robust read)
# Explicit schema + `mode=PERMISSIVE` means a malformed or wrong file (corrupted CSV, even a

from pyspark.sql import functions as F
WH_COLS = ["warehouse_event_id","warehouse_id","order_id","tracking_number","carrier_name",
           "origin_city","origin_country","warehouse_status","status_timestamp_raw",
           "record_updated_at","batch_date"]
wh_schema = ", ".join(f"{c} string" for c in WH_COLS) + ", _corrupt_record string"

(spark.read.option("header", True).option("sep", ";")
    .option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(wh_schema).csv(f"{BASE}/warehouse_updates.csv")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .createOrReplaceTempView("_raw_wh"))

spark.sql(f"""CREATE OR REPLACE TABLE {SCHEMA}.bronze_warehouse_rejected AS
  SELECT *, CASE WHEN _corrupt_record IS NOT NULL THEN 'unparseable_row' ELSE 'missing_key' END AS reject_reason,
         current_timestamp() AS _rejected_at
  FROM _raw_wh WHERE _corrupt_record IS NOT NULL OR warehouse_event_id IS NULL""")

spark.sql("""CREATE OR REPLACE TEMP VIEW _wh_good AS
  SELECT * FROM _raw_wh WHERE _corrupt_record IS NULL AND warehouse_event_id IS NOT NULL""")

# INCREMENTAL + BACKDATED: upsert by warehouse_event_id (first run creates the table)
if not spark.catalog.tableExists(f"{SCHEMA}.bronze_warehouse"):
    spark.sql(f"CREATE TABLE {SCHEMA}.bronze_warehouse AS SELECT * FROM _wh_good")
else:
    spark.sql(f"""
      MERGE INTO {SCHEMA}.bronze_warehouse AS t
      USING _wh_good AS s
        ON t.warehouse_event_id = s.warehouse_event_id
      WHEN MATCHED AND to_timestamp(s.record_updated_at,'dd/MM/yyyy HH:mm')
                     > to_timestamp(t.record_updated_at,'dd/MM/yyyy HH:mm')
           THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)

print("bronze_warehouse good:", spark.table(f"{SCHEMA}.bronze_warehouse").count(),
      "| rejected:", spark.table(f"{SCHEMA}.bronze_warehouse_rejected").count())

# COMMAND ----------
# %md ## 3 · BRONZE — carrier (STREAMING, Auto Loader with schema RESCUE)
# Auto Loader watches the folder and ingests new JSON files incrementally
# (`availableNow=True` = process what's here now, then stop; re-running picks up only NEW files).

from delta.tables import DeltaTable
from pyspark.sql import Window as W

def upsert_bronze_carrier(batch_df, batch_id):
    w = W.partitionBy("event_id").orderBy(F.col("ingested_at").desc())
    deduped = batch_df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")
    if not spark.catalog.tableExists(f"{SCHEMA}.bronze_carrier"):
        deduped.write.format("delta").saveAsTable(f"{SCHEMA}.bronze_carrier")
    else:
        (DeltaTable.forName(spark, f"{SCHEMA}.bronze_carrier").alias("t")
            .merge(deduped.alias("s"), "t.event_id = s.event_id")
            .whenMatchedUpdateAll(condition="s.ingested_at > t.ingested_at")
            .whenNotMatchedInsertAll()
            .execute())

q = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{BASE}/_schema/carrier")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("badRecordsPath", f"{BASE}/_bad_records/carrier")
    .option("multiLine", "true")
    .option("pathGlobFilter", "*carrier*.json")
    .load(BASE)
    .withColumn("_ingested_at", F.current_timestamp())
    .writeStream.option("checkpointLocation", f"{BASE}/_checkpoint/carrier")
    .foreachBatch(upsert_bronze_carrier)
    .trigger(availableNow=True).start())
q.awaitTermination()

print("bronze_carrier rows:", spark.table(f"{SCHEMA}.bronze_carrier").count())
rescued = spark.table(f"{SCHEMA}.bronze_carrier").filter("_rescued_data IS NOT NULL")
print("carrier rows with rescued (drifted) data:", rescued.count())

# COMMAND ----------
# %md ## 4 · GATE — Bronze
# Sources arrived, keys are present and unique. If not, STOP.

verify_data_quality("BRONZE", [
    # HARD: Stop ONLY if a totally broken feed arrives (0 good rows)
    ("warehouse_has_valid_rows", f"SELECT CASE WHEN count(*)=0 THEN 1 ELSE 0 END FROM {SCHEMA}.bronze_warehouse"),
    ("carrier_has_valid_rows",   f"SELECT CASE WHEN count(*)=0 THEN 1 ELSE 0 END FROM {SCHEMA}.bronze_carrier"),
    ("carrier_event_id_present", f"SELECT count(*) FROM {SCHEMA}.bronze_carrier WHERE event_id IS NULL"),
    
    # WARN: Log duplicates and unreadable rows for engineering review, but DO NOT STOP THE PIPELINE
    ("warehouse_id_duplicates",  f"SELECT count(*)-count(DISTINCT warehouse_event_id) FROM {SCHEMA}.bronze_warehouse", "WARN"),
    ("carrier_id_duplicates",    f"SELECT count(*)-count(DISTINCT event_id) FROM {SCHEMA}.bronze_carrier", "WARN"),
    ("warehouse_unreadable",     f"SELECT count(*) FROM {SCHEMA}.bronze_warehouse_rejected", "WARN")
])

In [0]:
# COMMAND ----------
# %md ## 5 · Lookup tables (seeds)

# 5a · carrier name cleanup
spark.createDataFrame(
    [("DHL","DHL"),("FEDEX","FedEx"),("UPS","UPS"),("ROYAL MAIL","Royal Mail"),
     ("AUSTRALIA POST","Australia Post"),("NINJA VAN","Ninja Van"),("POSTNL","PostNL"),
     ("ARAMEX","Aramex")], ["raw","carrier"]).createOrReplaceTempView("carrier_map")


# 5b · shared status vocabulary. status_rank = journey order; is_terminal = final state.
spark.createDataFrame(
    [("PROCESSING","processing",10,False),("PACKED","packed",20,False),
     ("RECEIVED","received",25,False),("SHIPPED","shipped",40,False),
     ("IN_TRANSIT","in_transit",50,False),("DELAYED","exception",55,False),
     ("EXCEPTION","exception",55,False),("OUT_FOR_DELIVERY","out_for_delivery",60,False),
     ("DELIVERED","delivered",90,True),("CANCELLED","cancelled",95,True),
     ("RETURNED","returned",95,True),("LABEL_CREATED","label_created",5,False),
     ("PICKED_UP","shipped",40,False)],
    ["raw","status","status_rank","is_terminal"]).createOrReplaceTempView("status_map")

# 5c · warehouse timezone (local -> UTC), with country as a fallback
spark.createDataFrame(
    [("WH-AMS","Europe/Amsterdam"),("WH-LHR","Europe/London"),("WH-LAX","America/Los_Angeles"),
     ("WH-SIN","Asia/Singapore"),("WH-SYD","Australia/Sydney"),("WH-DXB","Asia/Dubai")],
    ["warehouse_id","tz"]).createOrReplaceTempView("warehouse_tz")
spark.createDataFrame(
    [("NL","Europe/Amsterdam"),("GB","Europe/London"),("US","America/Los_Angeles"),
     ("SG","Asia/Singapore"),("AU","Australia/Sydney"),("AE","Asia/Dubai")],
    ["origin_country","tz"]).createOrReplaceTempView("country_tz")

In [0]:
# COMMAND ----------
# %md ## 6 · SILVER — clean warehouse, split GOOD vs REJECTED

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW _wh_typed AS
WITH cleaned AS (
  SELECT w.warehouse_event_id, w.warehouse_id, w.order_id,
    nullif(upper(trim(w.tracking_number)),'')           AS tracking_number,   -- upper: case-insensitive id
    coalesce(cm.carrier, upper(trim(w.carrier_name)))   AS carrier,
    w.origin_city, w.origin_country, sm.status, sm.status_rank, sm.is_terminal,
    w.warehouse_status                                  AS status_raw,
    coalesce(wtz.tz, ctz.tz, 'UTC')                     AS site_tz,
    to_utc_timestamp(to_timestamp(w.status_timestamp_raw,'dd/MM/yyyy HH:mm'),
                     coalesce(wtz.tz, ctz.tz, 'UTC'))   AS event_ts_utc,
    to_utc_timestamp(to_timestamp(w.record_updated_at,'dd/MM/yyyy HH:mm'),
                     coalesce(wtz.tz, ctz.tz, 'UTC'))   AS record_updated_at,
    w.batch_date
  FROM {SCHEMA}.bronze_warehouse w
  LEFT JOIN carrier_map  cm  ON upper(trim(w.carrier_name))     = cm.raw
  LEFT JOIN status_map   sm  ON upper(trim(w.warehouse_status)) = sm.raw
  LEFT JOIN warehouse_tz wtz ON w.warehouse_id = wtz.warehouse_id
  LEFT JOIN country_tz   ctz ON upper(trim(w.origin_country))   = ctz.origin_country
)
SELECT *,
  -- PRODUCTION GRAIN: (carrier, tracking_number); fall back to order key when tracking missing
  CASE WHEN tracking_number IS NOT NULL THEN concat(carrier,':',tracking_number)
       ELSE concat('WH:',warehouse_id,'|ORD:',order_id) END AS shipment_key,
  CASE WHEN status IS NULL       THEN 'unknown_status'
       WHEN event_ts_utc IS NULL THEN 'bad_timestamp'
       ELSE NULL END AS reject_reason
FROM cleaned
""")

# rejected rows -> quarantine table (kept, not dropped)
spark.sql(f"""
CREATE OR REPLACE TABLE {SCHEMA}.silver_warehouse_rejected AS
SELECT *, current_timestamp() AS _rejected_at FROM _wh_typed WHERE reject_reason IS NOT NULL
""")

# good rows -> silver, with dedup
spark.sql(f"""
CREATE OR REPLACE TABLE {SCHEMA}.silver_warehouse AS
WITH good AS (SELECT * FROM _wh_typed WHERE reject_reason IS NULL),
ranked AS (
  SELECT *, row_number() OVER (PARTITION BY shipment_key, status, event_ts_utc
                               ORDER BY record_updated_at DESC, warehouse_event_id DESC) AS dup_rank
  FROM good
)
SELECT warehouse_event_id, warehouse_id, order_id, tracking_number, carrier, origin_city,
       origin_country, status, status_rank, is_terminal, status_raw, site_tz, event_ts_utc,
       record_updated_at, batch_date, shipment_key
FROM ranked WHERE dup_rank = 1
""")
print("silver_warehouse good:", spark.table(f"{SCHEMA}.silver_warehouse").count(),
      "| rejected:", spark.table(f"{SCHEMA}.silver_warehouse_rejected").count())

# COMMAND ----------
# %md ## 7 · SILVER — clean carrier, split GOOD vs REJECTED

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW _ca_typed AS
SELECT c.event_id, coalesce(cm.carrier, upper(trim(c.carrier))) AS carrier,
  upper(trim(c.tracking_number)) AS tracking_number, sm.status, sm.status_rank, sm.is_terminal,
  c.event_type AS status_raw, to_timestamp(c.event_time) AS event_ts_utc,
  to_timestamp(c.ingested_at) AS ingested_at, c.location, c.event_timezone,
  concat(coalesce(cm.carrier, upper(trim(c.carrier))), ':', upper(trim(c.tracking_number))) AS shipment_key,
  CASE WHEN sm.status IS NULL                THEN 'unknown_status'
       WHEN to_timestamp(c.event_time) IS NULL THEN 'bad_timestamp'
       WHEN c.event_id IS NULL              THEN 'missing_event_id'
       ELSE NULL END AS reject_reason
FROM {SCHEMA}.bronze_carrier c
LEFT JOIN carrier_map cm ON upper(trim(c.carrier))    = cm.raw
LEFT JOIN status_map  sm ON upper(trim(c.event_type)) = sm.raw
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {SCHEMA}.silver_carrier_rejected AS
SELECT *, current_timestamp() AS _rejected_at FROM _ca_typed WHERE reject_reason IS NOT NULL
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {SCHEMA}.silver_carrier AS
WITH good AS (SELECT * FROM _ca_typed WHERE reject_reason IS NULL),
ranked AS (SELECT *, row_number() OVER (PARTITION BY event_id ORDER BY ingested_at DESC) AS dup_rank FROM good)
SELECT event_id, carrier, tracking_number, status, status_rank, is_terminal, status_raw,
       event_ts_utc, ingested_at, location, event_timezone, shipment_key
FROM ranked WHERE dup_rank = 1
""")
print("silver_carrier good:", spark.table(f"{SCHEMA}.silver_carrier").count(),
      "| rejected:", spark.table(f"{SCHEMA}.silver_carrier_rejected").count())

# COMMAND ----------
# %md ## 8 · SILVER — unified event stream  (table: silver_events)
# Both cleaned sources stacked into one event list (same columns).

spark.sql(f"""
CREATE OR REPLACE TABLE {SCHEMA}.silver_events AS
SELECT shipment_key, tracking_number, carrier, order_id, status, status_raw, status_rank, is_terminal,
       event_ts_utc, 'warehouse' AS source_system, cast(warehouse_event_id AS string) AS source_event_id,
       origin_city, origin_country, cast(null AS string) AS location, record_updated_at AS source_loaded_at
FROM {SCHEMA}.silver_warehouse
UNION ALL
SELECT shipment_key, tracking_number, carrier, cast(null AS string) AS order_id, status, status_raw, status_rank, is_terminal,
       event_ts_utc, 'carrier' AS source_system, event_id AS source_event_id,
       cast(null AS string), cast(null AS string), location, ingested_at
FROM {SCHEMA}.silver_carrier
""")
print("silver_events rows:", spark.table(f"{SCHEMA}.silver_events").count())

# COMMAND ----------
# %md ## 9 · GATE — Silver
# HARD checks (must be clean, or STOP): the GOOD silver tables have no duplicates, no nulls,
# valid statuses — because bad rows were already quarantined, these should pass.
# WARN checks (tracked, don't stop): how many rows we quarantined this run. A spike here is a

ACCEPTED = ("'label_created','processing','packed','received','shipped','in_transit',"
            "'exception','out_for_delivery','delivered','cancelled','returned'")
verify_data_quality("SILVER", [
    # HARD — the promoted (good) data must be clean
    ("warehouse_dupes_removed", f"SELECT count(*) FROM (SELECT shipment_key,status,event_ts_utc FROM {SCHEMA}.silver_warehouse GROUP BY 1,2,3 HAVING count(*)>1)"),
    ("carrier_event_id_unique", f"SELECT count(*)-count(DISTINCT event_id) FROM {SCHEMA}.silver_carrier"),
    ("good_status_mapped",      f"SELECT count(*) FROM {SCHEMA}.silver_events WHERE status IS NULL"),
    ("good_status_in_vocab",    f"SELECT count(*) FROM {SCHEMA}.silver_events WHERE status NOT IN ({ACCEPTED})"),
    ("good_carrier_present",    f"SELECT count(*) FROM {SCHEMA}.silver_events WHERE carrier IS NULL"),
    ("good_timestamps_parsed",  f"SELECT count(*) FROM {SCHEMA}.silver_events WHERE event_ts_utc IS NULL"),
    ("good_shipment_key",       f"SELECT count(*) FROM {SCHEMA}.silver_events WHERE shipment_key IS NULL"),
    # WARN — quarantined rows (kept in *_rejected, not promoted). Investigate if non-zero.
    ("warehouse_rejected",      f"SELECT count(*) FROM {SCHEMA}.silver_warehouse_rejected", "WARN"),
    ("carrier_rejected",        f"SELECT count(*) FROM {SCHEMA}.silver_carrier_rejected", "WARN"),
    ("carrier_schema_drift",    f"SELECT count(*) FROM {SCHEMA}.bronze_carrier WHERE _rescued_data IS NOT NULL", "WARN"),
])

In [0]:
# COMMAND ----------
# %md ## 10 · GOLD — shipment_status  (one row per shipment)
# Pick the winning event per shipment: 1) terminal state is sticky, 2) latest event time,
# 3) later stage, 4) carrier preferred. Add counts + review flags.

spark.sql(f"""
CREATE OR REPLACE TABLE {SCHEMA}.gold_shipment_status AS
WITH winning AS (
  SELECT * FROM (
    SELECT *, row_number() OVER (PARTITION BY shipment_key
      ORDER BY CASE WHEN is_terminal THEN 1 ELSE 0 END DESC, event_ts_utc DESC, status_rank DESC,
               CASE source_system WHEN 'carrier' THEN 0 ELSE 1 END, source_loaded_at DESC) AS pick
    FROM {SCHEMA}.silver_events) WHERE pick = 1
),
summary AS (
  SELECT shipment_key, count(*) AS event_count,
         count_if(source_system='warehouse') AS warehouse_event_count,
         count_if(source_system='carrier')   AS carrier_event_count,
         min(event_ts_utc) AS first_event_ts_utc, max(event_ts_utc) AS last_event_ts_utc,
         max(CASE WHEN NOT is_terminal THEN event_ts_utc END) AS last_nonterminal_ts_utc,
         max(CASE WHEN is_terminal AND source_system='carrier'   THEN true ELSE false END) AS carrier_terminal,
         max(CASE WHEN is_terminal AND source_system='warehouse' THEN true ELSE false END) AS warehouse_terminal,
         count(DISTINCT source_system) AS source_system_count,
         max(origin_city)    AS origin_city_any,     -- descriptive attrs from the shipment,
         max(origin_country) AS origin_country_any,   -- not the winning (carrier) status event
         count(DISTINCT carrier)  AS distinct_carrier_count,  -- tracking# unique only within a carrier
         count(DISTINCT order_id) AS distinct_order_count,    -- >1 carrier/order = colliding string
         max(CASE WHEN source_loaded_at < event_ts_utc THEN true ELSE false END) AS has_clock_anomaly
  FROM {SCHEMA}.silver_events GROUP BY shipment_key
)
SELECT w.shipment_key, w.tracking_number, (w.tracking_number IS NULL) AS missing_tracking_number,
       (s.distinct_carrier_count > 1 OR s.distinct_order_count > 1) AS ambiguous_tracking,
       s.distinct_carrier_count, s.distinct_order_count,
       CASE WHEN s.distinct_carrier_count>1 OR s.distinct_order_count>1 THEN NULL ELSE w.carrier END AS carrier,
       CASE WHEN s.distinct_carrier_count>1 OR s.distinct_order_count>1 THEN NULL ELSE s.origin_city_any END AS origin_city,
       CASE WHEN s.distinct_carrier_count>1 OR s.distinct_order_count>1 THEN NULL ELSE s.origin_country_any END AS origin_country,
       (s.origin_country_any IS NOT NULL AND s.distinct_carrier_count=1 AND s.distinct_order_count<=1) AS origin_known,
       w.status AS current_status,
       w.status_raw AS current_status_source_value, w.source_system AS current_status_source,
       w.event_ts_utc AS current_status_event_ts_utc, w.is_terminal,
       (s.source_system_count=2 AND s.carrier_terminal<>s.warehouse_terminal) AS has_status_conflict,
       (w.is_terminal AND s.last_nonterminal_ts_utc IS NOT NULL
                      AND s.last_nonterminal_ts_utc > w.event_ts_utc)          AS has_post_terminal_event,
       s.has_clock_anomaly,
       s.first_event_ts_utc, s.last_event_ts_utc, s.event_count,
       s.warehouse_event_count, s.carrier_event_count, s.source_system_count,
       current_timestamp() AS dwh_loaded_at_utc
FROM winning w JOIN summary s USING (shipment_key)
""")
print("gold_shipment_status rows:", spark.table(f"{SCHEMA}.gold_shipment_status").count())

# COMMAND ----------
# %md ## 11 · GATE — Gold
# The serving table is correct: exactly one row per shipment, key present, status valid.

verify_data_quality("GOLD", [
    ("one_row_per_shipment", f"SELECT count(*)-count(DISTINCT shipment_key) FROM {SCHEMA}.gold_shipment_status"),
    ("shipment_key_present", f"SELECT count(*) FROM {SCHEMA}.gold_shipment_status WHERE shipment_key IS NULL"),
    ("status_valid",         f"SELECT count(*) FROM {SCHEMA}.gold_shipment_status WHERE current_status NOT IN ({ACCEPTED})"),
])

# COMMAND ----------
# %md ## 12 · Result

display(spark.sql(f"""
SELECT count(*) AS shipments, count(DISTINCT shipment_key) AS distinct_keys,
       count_if(current_status='delivered')  AS delivered,
       count_if(missing_tracking_number)     AS missing_tracking,
       count_if(has_status_conflict)         AS conflicts,
       count_if(has_post_terminal_event)     AS post_terminal_anomalies
FROM {SCHEMA}.gold_shipment_status"""))
display(spark.table(f"{SCHEMA}.gold_shipment_status").orderBy("shipment_key"))

# COMMAND ----------
# %md ## 13 · Data-quality summary (run health)

display(spark.sql(f"""
  SELECT 'bronze' AS layer, 'warehouse' AS source, reject_reason, count(*) AS rows
    FROM {SCHEMA}.bronze_warehouse_rejected GROUP BY reject_reason
  UNION ALL SELECT 'silver','warehouse', reject_reason, count(*) FROM {SCHEMA}.silver_warehouse_rejected GROUP BY reject_reason
  UNION ALL SELECT 'silver','carrier',   reject_reason, count(*) FROM {SCHEMA}.silver_carrier_rejected GROUP BY reject_reason
  UNION ALL SELECT 'bronze','carrier','schema_drift_rescued', count(*) FROM {SCHEMA}.bronze_carrier WHERE _rescued_data IS NOT NULL
  ORDER BY layer, source, reject_reason
"""))
